In [68]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.svm import SVC

In [69]:
df = pd.read_csv("titanic.csv")

In [70]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [71]:
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], axis = 1)

In [72]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [73]:
le_sex = LabelEncoder()
le_Embarked = LabelEncoder()

In [74]:
df['Sex'] = le_sex.fit_transform(df['Sex'])
df['Embarked'] = le_Embarked.fit_transform(df['Embarked'])

In [75]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,2
1,1,1,0,38.0,1,0,71.2833,0
2,1,3,0,26.0,0,0,7.9250,2
3,1,1,0,35.0,1,0,53.1000,2
4,0,3,1,35.0,0,0,8.0500,2


In [76]:
df.shape

(891, 8)

In [77]:
df.isna().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      0
dtype: int64

In [78]:
df.duplicated().sum()

np.int64(111)

In [79]:
df = df.drop_duplicates()

In [80]:
df.duplicated().sum()

np.int64(0)

In [81]:
df['Age'] = df['Age'].fillna(df['Age'].mean())

In [82]:
df.isna().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

In [83]:
x = df.drop('Survived', axis=1)
y = df['Survived']

In [84]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= .2, random_state=42)

In [85]:
std = StandardScaler()

In [86]:
x_train = std.fit_transform(x_train)
x_test = std.transform(x_test)

In [87]:
model1_wo_grid = SVC()

In [88]:
model1_wo_grid.fit(x_train, y_train)
print(model1_wo_grid.score(x_train, y_train))
print(model1_wo_grid.score(x_test, y_test))
pre_y1 = model1_wo_grid.predict(x_test)
print(accuracy_score(pre_y1, y_test))

0.8269230769230769
0.8141025641025641
0.8141025641025641


In [90]:
svc = SVC()

In [103]:
param_grid = {
  'C' : [0.1, 1, 10, 100],
  'kernel' : ['linear', 'rbf', 'poly', 'sigmoid'],
  'gamma': ['scale', 'auto', 0.01, 0.1, 1]
  # 'degree' : [2,3,5,7]
}

In [104]:
grid_search = GridSearchCV(
  estimator=svc,
  param_grid=param_grid,
  cv = 5, n_jobs=-1, 
  verbose=10
  )

In [105]:
grid_model = grid_search.fit(x_train, y_train)
print(grid_model.score(x_train, y_train))
print(grid_model.score(x_test, y_test))
pre_y2 = grid_model.predict(x_test)
print(accuracy_score(pre_y2, y_test))

Fitting 5 folds for each of 80 candidates, totalling 400 fits
0.8413461538461539
0.7884615384615384
0.7884615384615384


In [106]:
grid_model.best_params_

{'C': 10, 'gamma': 0.1, 'kernel': 'rbf'}

In [95]:
from xgboost import XGBRFClassifier

In [96]:
xgc = XGBRFClassifier()

In [97]:
xgc.fit(x_train, y_train)
print(xgc.score(x_train, y_train))
print(xgc.score(x_test, y_test))

0.8589743589743589
0.7948717948717948


In [98]:
param_grid = {
  "n_estimators" : [100, 200, 300, 400],
  "learning_rate" : [.1, .001, 1, .5],
  "max_depth" : [3,5,7,9],
  "subsample" : [.6,.7,.8],
  "colsample_bytree" : [.7,.8,.9],
  "objective" : ['binary:logistic', "multi:softmax"]
}

In [100]:
grid_CV = GridSearchCV(estimator=xgc, param_grid=param_grid, n_jobs=-1, verbose=10 )

In [101]:
grid_CV.fit(x_train, y_train)
print(grid_CV.score(x_train, y_train))
print(grid_CV.score(x_test, y_test))

Fitting 5 folds for each of 1152 candidates, totalling 5760 fits


c:\Users\ABDULLAH AL MASUM\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
2880 fits failed out of a total of 5760.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2880 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\ABDULLAH AL MASUM\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\ABDULLAH AL MASUM\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\core.py", line 751, in inner_f
    return func(**kwargs)
  File "c:\Users\ABDULLAH AL MASUM\AppData\Local\Programs\Pyth

0.8717948717948718
0.7884615384615384
